# Variant-effect predictors across membrane protein families

Representatives of the largest integral-membrane Pfam families (MMseqs2 cluster representatives, see `scripts/family_sets.py`) scored with every predictor and compared family by family. There are no clinical labels, so the questions are about the predictors themselves: how much they agree, how widely they spread their scores, how strongly they penalise transmembrane positions, and which protein properties go with disagreement. Figures go to `figures_families/`, tables to `figures_families/tables/`.

In [ ]:
import os, sys, warnings
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import Markdown, display
sys.path.insert(0, str(Path.cwd().parent))
warnings.filterwarnings("ignore", category=RuntimeWarning)
from mvsa import talk as T, families as F
from mvsa.data import load, available_predictors
from mvsa.groups import MAIN_PREDICTORS

DATA_DIR = Path(os.environ.get("MVSA_DATA", "../data_families"))
FIG_DIR = Path(os.environ.get("MVSA_FIGURES", "../figures_families"))
MEMBERSHIP = Path(os.environ.get("MVSA_MEMBERSHIP", "../../sets/families/family_membership.tsv"))
MIN_PER_FAMILY = 20          # families with fewer scored proteins are left out of the comparisons

T.style(FIG_DIR)
TABLE_DIR = FIG_DIR / "tables"; TABLE_DIR.mkdir(parents=True, exist_ok=True)
D = load(DATA_DIR)
mem = F.membership(MEMBERSHIP)
PCORR = pd.read_parquet(DATA_DIR / "protein_correlations.parquet") if (DATA_DIR / "protein_correlations.parquet").exists() else None
PRED = [p for p in MAIN_PREDICTORS if p in available_predictors(D["variant_sample"])]
t = F.protein_table(D, PCORR, mem, PRED)
keep = t.family.value_counts()
t = t[t.family.isin(keep[keep >= MIN_PER_FAMILY].index)]
LAB = {"length": "length", "n_tm_helical": "TM helices", "plddt_mean": "pLDDT (whole chain)", "frac_excluded": "fraction disordered/excluded",
       "plddt_trimmed": "pLDDT (structured part)", "frac_helix": "helix fraction", "inversion_index_kd": "hydrophobic surface (inversion)",
       "h_depth_slope_kd": "hydrophobicity rises to core", "belt_contrast_scan_kd": "hydrophobic belt", "contact_density_ca": "packing",
       "cavity_frac": "cavities", "a_over_c": "elongation", "sphericity_hull": "sphericity"}

def pair_label(c):
    a, b = c[4:].split("__")
    return f"{T.name(a)} – {T.name(b)}"

def note(*lines):
    display(Markdown("**Slide notes**\n\n" + "\n".join(f"- {l}" for l in lines)))

t = t.drop(columns=[c for c in t if (c.startswith('rho_') or c.startswith('dr_') or c.startswith('tm_')) and t[c].isna().all()])
KEY = F.main_pair(t)          # ESM-1v - ThermoMPNN agreement, or the next sequence-vs-stability pair with data
print(f"{t.accession.nunique()} proteins in {t.family.nunique()} families | predictors: {', '.join(PRED)} | "
      f"per-protein correlations: {PCORR is not None}")
if PCORR is None:
    print("protein_correlations.parquet missing: run lsf/protcorr.sh families first; agreement sections are skipped")

## 1. The families

In [ ]:
summ = F.family_summary(t, [f'dr_{p}' for p in PRED] + [c for c in t if c.startswith('rho_') or c.startswith('tm_')] + F.SHAPE + F.PROTEIN + ['plddt_trimmed'])
dom = pd.crosstab(t.family, t.domain.fillna('unknown'), normalize='index')
order = summ.sort_values('n_proteins').index.tolist()
fig, ax = plt.subplots(figsize=(11, 0.38 * len(order) + 1.8))
left = np.zeros(len(order))
for k, col in (('Bacteria', T.SOL), ('Archaea', '#8a9a5b'), ('Eukaryota', T.MP), ('unknown', T.LIGHT)):
    if k in dom:
        v = dom.reindex(order)[k].fillna(0).to_numpy(); ax.barh(range(len(order)), v, left=left, color=col, label=k); left += v
ax.set_yticks(range(len(order)), [f"{f}  ({int(summ.loc[f, 'n_proteins'])}, {summ.loc[f, 'n_tm_helical']:.0f} TM)" for f in order], fontsize=11)
ax.xaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1.0)); ax.set_xlabel('Share of representatives')
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.12), ncol=4)
T.headline(fig, f'{len(order)} membrane superfamilies, {t.accession.nunique():,} representative proteins',
           'Domain of life of the representatives; brackets: proteins scored, median TM helices')
T.save(fig, 'F1_families'); plt.show()
summ.round(3).to_csv(TABLE_DIR / 'family_summary.tsv', sep='\t')
t.to_csv(TABLE_DIR / 'family_proteins.tsv', sep='\t', index=False)
note(f"Multipass by UniProt annotation: {(t.class_annotation == 'MULTIPASS').mean():.0%} of representatives (the rest are mostly unannotated TrEMBL topology)",
     *[f'{f}: {int(r.n_proteins)} proteins, {r.frac_eukaryota:.0%} eukaryotic, median length {r.length:.0f}' for f, r in summ.sort_values('n_proteins', ascending=False).iterrows()])

## 2. How much do the predictors agree, family by family?

Median per-protein Spearman ρ between two predictors over all substitutions. Rows are families, sorted by ESM-1v – ThermoMPNN agreement; the colour shows how each family compares with the others for the same pair (z-score within the column).

In [ ]:
rho_cols = [c for c in summ if c.startswith('rho_')]
if rho_cols and KEY:
    key = KEY
    m = summ[rho_cols].sort_values(key)
    m.columns = [pair_label(c) for c in m.columns]
    fig = T.matrix(m, cbar='z-score within pair')
    lo_, hi_ = m.index[0], m.index[-1]
    col = pair_label(key)
    T.headline(fig, f'Sequence and stability predictors agree least in {lo_} and most in {hi_}',
               f'{col}: median ρ {m.loc[lo_, col]:.2f} vs {m.loc[hi_, col]:.2f}; numbers are medians over each family\'s proteins')
    T.save(fig, 'F2_agreement_matrix'); plt.show()
    fig = T.family_boxes(t, key, f'Spearman ρ, {pair_label(key)}, per protein')
    spread = t.groupby('family')[key].median()
    T.headline(fig, f'{pair_label(key)} agreement ranges from {spread.min():.2f} to {spread.max():.2f} across families',
               'Each point is one representative protein; boxes show the interquartile range')
    T.save(fig, 'F3_thermo_esm_by_family'); plt.show()
    note(*[f'{pair_label(c)}: family medians {summ[c].min():.2f}–{summ[c].max():.2f} (lowest {summ[c].idxmin()}, highest {summ[c].idxmax()})' for c in rho_cols if summ[c].notna().any()])

## 3. Score spread and how strongly TM positions are penalised

In [ ]:
dr = summ[[f'dr_{p}' for p in PRED if f'dr_{p}' in summ]].sort_values(f'dr_esm1v' if 'dr_esm1v' in summ else f'dr_{PRED[0]}')
dr.columns = [T.name(c[3:]) for c in dr.columns]
fig = T.matrix(dr, fmt='{:.2g}', cbar='z-score within predictor')
T.headline(fig, 'How widely each predictor spreads its scores, by family',
           'Median per-protein 5–95% range of oriented scores; a narrow range means the predictor barely separates substitutions')
T.save(fig, 'F4_dynamic_range'); plt.show()
tm = summ[[f'tm_{p}' for p in PRED if f'tm_{p}' in summ]]
if len(tm.columns):
    tm = tm.sort_values(tm.columns[0]); tm.columns = [T.name(c[3:]) for c in tm.columns]
    fig = T.matrix(tm, fmt='{:+.2g}', cbar='z-score within predictor')
    T.headline(fig, 'How much more damaging each predictor finds TM substitutions',
               'Median per protein of (median score in TM segments − median score outside); positive = TM penalised')
    T.save(fig, 'F5_tm_penalty'); plt.show()

## 4. Structure of each family

In [ ]:
sc = [c for c in F.SHAPE if c in summ and c != 'plddt_mean'] + (['plddt_trimmed'] if 'plddt_trimmed' in summ else [])
sm = summ[sc].sort_values('inversion_index_kd' if 'inversion_index_kd' in sc else sc[0])
sm.columns = [LAB.get(c, c) for c in sm.columns]
fig = T.matrix(sm, fmt='{:.2g}', cbar='z-score within metric')
T.headline(fig, 'Structural fingerprint of each family', 'Median over representatives (trimmed chains)')
T.save(fig, 'F6_structure'); plt.show()

## 5. What goes with disagreement?

Correlation between ESM-1v – ThermoMPNN agreement and protein properties: pooled over all proteins (open), and the median of the correlations computed inside each family (filled). Only the within-family value is free of differences between families.

In [ ]:
key = KEY
if key:
    feats = F.PROTEIN + [c for c in F.SHAPE if c != 'plddt_mean'] + ['plddt_trimmed', 'cluster_size'] + [f'dr_{p}' for p in PRED]
    drv = F.drivers(t, key, feats)
    assert len(drv), 'too few proteins per family to compute within-family correlations'
    fig = T.drivers_dots(drv, {**LAB, 'cluster_size': 'cluster size', **{f'dr_{p}': f'{T.name(p)} range' for p in PRED}})
    top = drv.iloc[0]
    T.headline(fig, f"Within families, agreement goes with {LAB.get(top.feature, top.feature)} (ρ = {top.rho_within_median:+.2f})",
               f'Spearman ρ between {pair_label(key)} agreement and each protein property')
    T.save(fig, 'F7_agreement_drivers'); plt.show()
    drv.round(3).to_csv(TABLE_DIR / 'agreement_drivers.tsv', sep='\t', index=False)
    de = F.domain_effect(t, key)
    if len(de):
        de['diff'] = de.eukaryota - de.bacteria
        display(de.sort_values('diff').round(3).style.hide(axis='index'))
        note(f"Eukaryotic minus bacterial agreement: median {de['diff'].median():+.2f} across {len(de)} families with both")

## 6. The least-agreeing proteins in each family

In [ ]:
if KEY:
    key = KEY
    worst = t.sort_values(key).groupby('family').head(5)
    cols = [c for c in ['family', 'accession', 'gene', 'protein_name', 'domain', 'kingdom', 'length', 'n_tm_helical', key,
                        F.pair_col('esm1v', 'saprot'), F.pair_col('thermompnn', 'solvent_exposure'), 'plddt_mean', 'frac_excluded'] if c in worst]
    worst[cols].to_csv(TABLE_DIR / 'least_agreeing_per_family.tsv', sep='\t', index=False)
    display(worst[cols].round(3).style.hide(axis='index'))